In [1]:
import torch

In [62]:
from transformers import AutoTokenizer, LlamaConfig, LlamaForCausalLM

config = LlamaConfig.from_pretrained("JackFram/llama-68m")
CACHE_DIR = '/rscratch/rishabhtiwari/cache'
draft = LlamaForCausalLM.from_pretrained(
pretrained_model_name_or_path='JackFram/llama-68m',
config=config,
cache_dir=CACHE_DIR,
torch_dtype=torch.bfloat16,
device_map="cuda:6"
)
target = LlamaForCausalLM.from_pretrained(
pretrained_model_name_or_path='JackFram/llama-68m',
config=config,
cache_dir=CACHE_DIR,
torch_dtype=torch.bfloat16,
device_map="cuda:6"
)
tokenizer = AutoTokenizer.from_pretrained(
    'JackFram/llama-68m', 
    use_fast=True, 
    legacy=False,
    tokenizer_type='llama')


In [63]:

input_ids = tokenizer.encode('Hello, world!, it lam', return_tensors='pt').to('cuda:6')

print(input_ids)

tensor([[    1, 15043, 29892,  3186, 18565,   372,   301,   314]],
       device='cuda:6')


In [64]:
print(input_ids.shape)

torch.Size([1, 8])


In [65]:
draft_output = draft(input_ids, use_cache=True)
target_output = target(input_ids, use_cache=True)



In [66]:
draft_output.past_key_values

((tensor([[[[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
            [-2.1562,  3.4062,  1.8359,  ..., -0.6914, -0.7578, -0.7891],
            [-0.4844,  1.2422,  2.1562,  ..., -0.6953, -1.3203, -1.4609],
            ...,
            [ 1.0625, -1.4219, -1.4297,  ..., -1.3203, -2.0625, -0.5703],
            [-0.9375, -1.2578, -1.5703,  ..., -0.9688, -1.6562, -0.6875],
            [ 0.7188,  0.1348,  0.2363,  ..., -1.6328, -1.8906, -0.7539]],
  
           [[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
            [ 0.5234, -0.1133,  0.9766,  ..., -0.0574,  0.8047,  1.8438],
            [ 0.1182,  0.1162,  0.1758,  ...,  0.8281,  0.4668,  0.7578],
            ...,
            [-1.2578,  0.6484, -1.0312,  ...,  0.6992,  0.7109,  0.3203],
            [-0.4395, -2.2031, -0.3730,  ..., -1.2891, -1.4531,  0.9922],
            [ 1.9922, -2.1875, -0.4629,  ...,  0.0557, -0.7969,  0.4336]],
  
           [[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.000

In [67]:
input_ids[:,1:2].shape

torch.Size([1, 1])

In [68]:
draft_output_2 = draft(input_ids[:,1:2], use_cache=True, past_key_values=draft_output.past_key_values)

target_output_2 = target(input_ids[:,1:],use_cache=True, past_key_values=draft_output.past_key_values)

In [69]:
draft_output_2.logits.shape

torch.Size([1, 1, 32000])

In [70]:
target_output_2.logits.shape

torch.Size([1, 7, 32000])

In [71]:
torch.mean(draft_output_2.logits - target_output_2.logits[:, :1, :])

tensor(-0.0093, device='cuda:6', grad_fn=<MeanBackward0>)